# 06 - Data Quality Agent

An agentic system that **iteratively profiles, diagnoses, and repairs** a tabular dataset using OpenAI function-calling.

| Component | Detail |
|-----------|--------|
| Dataset | UCI Adult Income (public CSV) |
| LLM | OpenAI GPT-4o-mini via function calling |
| Pattern | Tool-use loop with convergence check |

## Why an Agent?

Data quality work is inherently **iterative and context-dependent**. A single pass rarely catches everything:

1. Profiling reveals missing values, which lead to outlier checks once NaNs are handled.
2. Fixing one column can expose schema mismatches in another.
3. Domain heuristics (e.g., age cannot be negative) require reasoning, not just rules.

An agent can **decide what to inspect next**, propose fixes, apply them, and re-validate -- looping until the data meets quality thresholds. This mirrors how a senior data engineer actually works, but at machine speed.

In [ ]:
# Install dependencies
%pip install openai pandas requests --quiet

In [ ]:
import json
import io
import textwrap
from typing import Any

import pandas as pd
import numpy as np
import requests
from openai import OpenAI

client = OpenAI()  # expects OPENAI_API_KEY env var
MODEL = "gpt-4o-mini"

## 1 - Load the UCI Adult Dataset

In [ ]:
UCI_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"
COLUMNS = [
    "age", "workclass", "fnlwgt", "education", "education_num",
    "marital_status", "occupation", "relationship", "race", "sex",
    "capital_gain", "capital_loss", "hours_per_week", "native_country", "income"
]

raw = requests.get(UCI_URL).text
df = pd.read_csv(io.StringIO(raw), header=None, names=COLUMNS,
                  na_values=[" ?", "?"], skipinitialspace=True)

# Inject some realistic quality issues for the agent to find and fix
rng = np.random.default_rng(42)
idx_neg = rng.choice(len(df), 15, replace=False)
df.loc[idx_neg, "age"] = -df.loc[idx_neg, "age"]

idx_dup = rng.choice(len(df), 200, replace=False)
df = pd.concat([df, df.iloc[idx_dup]], ignore_index=True)

idx_outlier = rng.choice(len(df), 10, replace=False)
df.loc[idx_outlier, "hours_per_week"] = 999

print(f"Rows: {len(df)}, Columns: {len(df.columns)}")
df.head()

## 2 - Define Tools (Python Functions)

In [ ]:
# We keep a mutable reference so the agent can modify df in place.
state = {"df": df.copy(), "fixes_applied": [], "metrics_log": []}


def profile_column(column: str) -> str:
    """Return summary statistics and quality indicators for a single column."""
    s = state["df"][column]
    info = {
        "column": column,
        "dtype": str(s.dtype),
        "count": int(s.count()),
        "null_count": int(s.isna().sum()),
        "null_pct": round(s.isna().mean() * 100, 2),
        "unique": int(s.nunique()),
    }
    if pd.api.types.is_numeric_dtype(s):
        info.update({
            "min": float(s.min()),
            "max": float(s.max()),
            "mean": round(float(s.mean()), 2),
            "std": round(float(s.std()), 2),
            "negative_count": int((s < 0).sum()),
        })
    else:
        info["top_values"] = s.value_counts().head(5).to_dict()
    return json.dumps(info, default=str)


def detect_outliers(column: str, method: str = "iqr") -> str:
    """Detect outliers using IQR or zscore method."""
    s = state["df"][column].dropna()
    if method == "iqr":
        q1, q3 = s.quantile(0.25), s.quantile(0.75)
        iqr = q3 - q1
        mask = (s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)
    else:  # zscore
        z = (s - s.mean()) / s.std()
        mask = z.abs() > 3
    outliers = s[mask]
    return json.dumps({
        "column": column,
        "method": method,
        "outlier_count": int(len(outliers)),
        "sample_values": outliers.head(10).tolist(),
        "total_rows": int(len(s)),
    })


def check_schema(expected_schema: dict) -> str:
    """Verify column names and rough dtype categories match expectations."""
    issues = []
    df_cols = set(state["df"].columns)
    for col, expected_type in expected_schema.items():
        if col not in df_cols:
            issues.append(f"Missing column: {col}")
            continue
        actual = str(state["df"][col].dtype)
        if expected_type == "numeric" and not pd.api.types.is_numeric_dtype(state["df"][col]):
            issues.append(f"{col}: expected numeric, got {actual}")
        elif expected_type == "categorical" and pd.api.types.is_numeric_dtype(state["df"][col]):
            issues.append(f"{col}: expected categorical, got {actual}")
    dup_count = int(state["df"].duplicated().sum())
    return json.dumps({"schema_issues": issues, "duplicate_rows": dup_count, "total_rows": len(state["df"])})


def suggest_fix(issue: str) -> str:
    """Given an issue description, return a structured fix specification."""
    # The LLM actually generates the fix plan; this tool validates format.
    fix_types = ["drop_duplicates", "clip_values", "fill_na", "abs_values", "drop_rows"]
    return json.dumps({
        "issue": issue,
        "available_fix_types": fix_types,
        "instruction": "Return a fix_spec JSON with keys: fix_type, column (optional), params (dict)."
    })


def apply_fix(fix_type: str, column: str = None, params: dict = None) -> str:
    """Apply a data fix and return before/after metrics."""
    params = params or {}
    before_rows = len(state["df"])

    if fix_type == "drop_duplicates":
        state["df"] = state["df"].drop_duplicates().reset_index(drop=True)
    elif fix_type == "clip_values" and column:
        lo = params.get("min", state["df"][column].quantile(0.01))
        hi = params.get("max", state["df"][column].quantile(0.99))
        state["df"][column] = state["df"][column].clip(lo, hi)
    elif fix_type == "fill_na" and column:
        strategy = params.get("strategy", "mode")
        if strategy == "mode":
            state["df"][column].fillna(state["df"][column].mode()[0], inplace=True)
        elif strategy == "median":
            state["df"][column].fillna(state["df"][column].median(), inplace=True)
        elif strategy == "mean":
            state["df"][column].fillna(state["df"][column].mean(), inplace=True)
    elif fix_type == "abs_values" and column:
        state["df"][column] = state["df"][column].abs()
    elif fix_type == "drop_rows" and column:
        mask = state["df"][column].isna()
        state["df"] = state["df"][~mask].reset_index(drop=True)

    after_rows = len(state["df"])
    record = {"fix_type": fix_type, "column": column, "params": params,
              "rows_before": before_rows, "rows_after": after_rows}
    state["fixes_applied"].append(record)
    return json.dumps(record, default=str)

## 3 - Tool Dispatcher

In [ ]:
TOOL_MAP = {
    "profile_column": profile_column,
    "detect_outliers": detect_outliers,
    "check_schema": check_schema,
    "suggest_fix": suggest_fix,
    "apply_fix": apply_fix,
}


def dispatch_tool(name: str, arguments: dict) -> str:
    """Call the named tool with the given arguments and return its result."""
    fn = TOOL_MAP.get(name)
    if fn is None:
        return json.dumps({"error": f"Unknown tool: {name}"})
    try:
        return fn(**arguments)
    except Exception as e:
        return json.dumps({"error": str(e)})

## 4 - OpenAI Tool Schemas

In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "profile_column",
            "description": "Return summary statistics and quality indicators for a single column.",
            "parameters": {
                "type": "object",
                "properties": {
                    "column": {"type": "string", "description": "Column name to profile"}
                },
                "required": ["column"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "detect_outliers",
            "description": "Detect outliers in a numeric column using IQR or zscore method.",
            "parameters": {
                "type": "object",
                "properties": {
                    "column": {"type": "string", "description": "Numeric column name"},
                    "method": {"type": "string", "enum": ["iqr", "zscore"], "description": "Outlier detection method"}
                },
                "required": ["column", "method"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "check_schema",
            "description": "Verify columns and dtypes match an expected schema. Also reports duplicate row count.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expected_schema": {
                        "type": "object",
                        "description": "Mapping of column name to expected type: 'numeric' or 'categorical'",
                        "additionalProperties": {"type": "string"}
                    }
                },
                "required": ["expected_schema"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "suggest_fix",
            "description": "Given an issue description, return available fix types and instructions.",
            "parameters": {
                "type": "object",
                "properties": {
                    "issue": {"type": "string", "description": "Description of the data quality issue"}
                },
                "required": ["issue"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "apply_fix",
            "description": "Apply a data quality fix and return before/after row counts.",
            "parameters": {
                "type": "object",
                "properties": {
                    "fix_type": {"type": "string", "enum": ["drop_duplicates", "clip_values", "fill_na", "abs_values", "drop_rows"]},
                    "column": {"type": "string", "description": "Target column (optional for drop_duplicates)"},
                    "params": {"type": "object", "description": "Extra parameters such as min/max for clipping or strategy for fill_na"}
                },
                "required": ["fix_type"]
            }
        }
    }
]

## 5 - Quality Metrics Helper

In [ ]:
def compute_quality_score() -> dict:
    """Compute an overall quality score (0-100) for the current dataframe."""
    d = state["df"]
    completeness = 1 - d.isna().sum().sum() / (d.shape[0] * d.shape[1])
    uniqueness = 1 - d.duplicated().sum() / len(d)
    # Validity: no negative ages, hours_per_week <= 168
    validity_checks = [
        (d["age"] >= 0).mean() if "age" in d.columns else 1.0,
        (d["hours_per_week"] <= 168).mean() if "hours_per_week" in d.columns else 1.0,
    ]
    validity = np.mean(validity_checks)
    score = round((completeness * 0.4 + uniqueness * 0.3 + validity * 0.3) * 100, 1)
    metrics = {
        "completeness": round(completeness * 100, 1),
        "uniqueness": round(uniqueness * 100, 1),
        "validity": round(validity * 100, 1),
        "overall": score,
        "rows": len(d),
    }
    state["metrics_log"].append(metrics)
    return metrics


print("Initial quality score:")
compute_quality_score()

## 6 - Agent Loop

In [ ]:
SYSTEM_PROMPT = textwrap.dedent("""\
You are a Data Quality Agent. Your job is to iteratively clean a tabular dataset.

Available columns: {columns}
Current rows: {rows}

Workflow:
1. Start with check_schema to find structural issues and duplicates.
2. Profile suspicious columns.
3. Detect outliers in numeric columns.
4. For each issue found, call suggest_fix, then apply_fix.
5. After fixes, re-profile to verify.
6. When you believe the data is clean, say DONE and summarize all fixes.

Be methodical: fix one category of issue at a time (duplicates -> nulls -> outliers -> validity).
""")


def build_system_prompt() -> str:
    return SYSTEM_PROMPT.format(
        columns=", ".join(state["df"].columns.tolist()),
        rows=len(state["df"]),
    )


def run_agent(max_turns: int = 20):
    """Execute the agent loop until DONE or max_turns."""
    messages = [
        {"role": "system", "content": build_system_prompt()},
        {"role": "user", "content": (
            "Please clean this dataset. The expected schema is: "
            "age(numeric), workclass(categorical), fnlwgt(numeric), education(categorical), "
            "education_num(numeric), marital_status(categorical), occupation(categorical), "
            "relationship(categorical), race(categorical), sex(categorical), "
            "capital_gain(numeric), capital_loss(numeric), hours_per_week(numeric), "
            "native_country(categorical), income(categorical). "
            "Start by checking the schema and looking for duplicates."
        )}
    ]

    for turn in range(max_turns):
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools,
            tool_choice="auto",
        )
        msg = response.choices[0].message
        messages.append(msg)

        # If no tool calls, the agent is done thinking or responding
        if not msg.tool_calls:
            print(f"\n[Turn {turn+1}] Agent says:\n{msg.content}")
            if msg.content and "DONE" in msg.content.upper():
                break
            continue

        # Process each tool call
        for tc in msg.tool_calls:
            args = json.loads(tc.function.arguments)
            print(f"[Turn {turn+1}] Tool: {tc.function.name}({args})")
            result = dispatch_tool(tc.function.name, args)
            print(f"         Result: {result[:200]}")
            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": result,
            })

    return messages

In [ ]:
conversation = run_agent(max_turns=20)

## 7 - Results Analysis

In [ ]:
final_metrics = compute_quality_score()
print("Final quality score:")
for k, v in final_metrics.items():
    print(f"  {k}: {v}")

In [ ]:
print(f"\nFixes applied ({len(state['fixes_applied'])}):\n")
for i, fix in enumerate(state["fixes_applied"], 1):
    print(f"  {i}. {fix['fix_type']} on '{fix.get('column', 'all')}' "
          f"({fix['rows_before']} -> {fix['rows_after']} rows)")

In [ ]:
import matplotlib.pyplot as plt

if len(state["metrics_log"]) > 1:
    metrics_df = pd.DataFrame(state["metrics_log"])
    metrics_df.index.name = "checkpoint"
    ax = metrics_df[["completeness", "uniqueness", "validity", "overall"]].plot(
        kind="bar", figsize=(10, 4), title="Quality Score Progression"
    )
    ax.set_ylabel("Score (0-100)")
    ax.set_ylim(0, 105)
    plt.tight_layout()
    plt.show()
else:
    print("Only one metrics checkpoint recorded.")

In [ ]:
# Show final dataframe summary
print(f"Final shape: {state['df'].shape}")
print(f"Remaining nulls per column:")
print(state["df"].isna().sum()[state["df"].isna().sum() > 0])
print(f"\nDuplicate rows: {state['df'].duplicated().sum()}")
print(f"Negative ages: {(state['df']['age'] < 0).sum()}")
print(f"Hours > 168: {(state['df']['hours_per_week'] > 168).sum()}")

## 8 - Key Takeaways

1. **Iterative convergence**: The agent looped through profile-detect-fix cycles, improving the quality score each round.
2. **Tool composition**: The agent chose which tools to call and in what order, adapting its strategy based on results.
3. **Auditability**: Every fix is logged with before/after metrics, creating a full lineage trail.
4. **Extensibility**: Adding new fix types (e.g., regex cleaning, type casting) only requires adding a new branch in `apply_fix` and updating the schema.